# Pazarlama Kampanyası Müşteri Kümeleme Modeli

Bu çalışmada bir pazarlama kampanyası veri seti ayrıntılı biçimde temizlenmiş, yaş ve toplam harcama gibi yeni özellikler türetilmiş, müşteriler benzer davranış özelliklerine göre gruplara ayrılmış; K-Means, Hiyerarşik Kümeleme, DBSCAN ve Gauss Karışım Modeli yöntemleri PCA ve t-SNE görselleştirmeleriyle karşılaştırılmış, son olarak her kümenin demografik ve harcama profili çıkarılmıştır.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, silhouette_samples
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('/content/marketing_campaign.csv', sep='\t')
print(f'Veri seti boyutu: {df.shape}')
print(f'Sütun sayısı: {len(df.columns)}')
print(f'Satır sayısı: {len(df)}')

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
copy_df = df.copy()
copy_df = copy_df.dropna()

In [ ]:
copy_df.info()

In [ ]:
copy_df.duplicated().sum()

In [ ]:
print('Eksik değer sayıları:')
print(copy_df.isnull().sum()[copy_df.isnull().sum() > 0])
print(f'Toplam eksik değer: {copy_df.isnull().sum().sum()}')

In [ ]:
df['Education'].unique()

In [ ]:
egitim_haritasi = {
    'Basic': 0,
    'Graduation': 1,
    '2n Cycle': 2,
    'PhD': 3,
    'Master': 4
}

copy_df['Education'] = copy_df['Education'].map(egitim_haritasi)

print(copy_df['Education'].value_counts())

In [ ]:
df['Marital_Status'].unique()

In [ ]:
birliktelik_haritasi = {
    'Single': 0,
    'Together': 1,
    'Married': 2,
    'Divorced': 3,
    'Widow': 4,
    'Alone': 1,
    'Absurd': 1,
    'YOLO': 1
}

copy_df['Marital_Status'] = copy_df['Marital_Status'].map(birliktelik_haritasi)

print(copy_df['Marital_Status'].value_counts())

In [ ]:
copy_df.head()

In [ ]:
copy_df['Dt_Customer'] = pd.to_datetime(copy_df['Dt_Customer'], format='%d-%m-%Y', dayfirst=True)

copy_df['Customer_Year'] = copy_df['Dt_Customer'].dt.year
copy_df['Customer_Month'] = copy_df['Dt_Customer'].dt.month
copy_df['Customer_Day'] = copy_df['Dt_Customer'].dt.day

print(copy_df[['Dt_Customer', 'Customer_Year', 'Customer_Month', 'Customer_Day']].head())

In [ ]:
copy_df.drop('Dt_Customer', axis=1, inplace=True)

In [ ]:
copy_df.head()

## Yeni Özellik Türetme

Analiz derinliğini artırmak amacıyla doğum yılından yaş bilgisi, harcama sütunlarının toplamından toplam harcama, çocuk sayısı sütunlarının toplamından toplam çocuk sayısı ve satın alma kanallarının toplamından toplam satın alma sayısı türetilmiştir.

In [ ]:
copy_df['Age'] = 2024 - copy_df['Year_Birth']

harcama_sutunlari = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
copy_df['Total_Spending'] = copy_df[harcama_sutunlari].sum(axis=1)

copy_df['Total_Children'] = copy_df['Kidhome'] + copy_df['Teenhome']

satin_alma_sutunlari = ['NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
copy_df['Total_Purchases'] = copy_df[satin_alma_sutunlari].sum(axis=1)

copy_df[['Age', 'Total_Spending', 'Total_Children', 'Total_Purchases']].describe()

## Türetilmiş Özelliklerin Keşifsel Analizi

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

sns.histplot(copy_df['Age'], bins=30, kde=True, ax=axes[0], color='#2980b9')
axes[0].set_title('Yaş Dağılımı')

sns.histplot(copy_df['Income'], bins=30, kde=True, ax=axes[1], color='#27ae60')
axes[1].set_title('Gelir Dağılımı')

sns.histplot(copy_df['Total_Spending'], bins=30, kde=True, ax=axes[2], color='#c0392b')
axes[2].set_title('Toplam Harcama Dağılımı')

sns.histplot(copy_df['Total_Purchases'], bins=30, kde=True, ax=axes[3], color='#8e44ad')
axes[3].set_title('Toplam Satın Alma Sayısı Dağılımı')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(data=copy_df, x='Education', y='Total_Spending', ax=axes[0], palette='Set2')
axes[0].set_title('Eğitim Seviyesine Göre Toplam Harcama')

sns.boxplot(data=copy_df, x='Marital_Status', y='Total_Spending', ax=axes[1], palette='Set3')
axes[1].set_title('Medeni Duruma Göre Toplam Harcama')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.scatterplot(data=copy_df, x='Income', y='Total_Spending', hue='Total_Children', palette='viridis', alpha=0.7)
plt.title('Gelir ile Toplam Harcama İlişkisi')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
onemli_sutunlar = ['Age', 'Income', 'Total_Spending', 'Total_Children', 'Total_Purchases', 'Recency', 'NumWebVisitsMonth']
sns.heatmap(copy_df[onemli_sutunlar].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Türetilmiş ve Temel Özellikler Arası Korelasyon Matrisi')
plt.tight_layout()
plt.show()

## Ölçeklendirme ve Boyut İndirgeme Hazırlığı

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(copy_df)
X_scaled = pd.DataFrame(X_scaled, columns=copy_df.columns, index=copy_df.index)

print('Ölçeklendirilmiş veri (ilk 5 satır):')
X_scaled.head()

## Optimum Küme Sayısının Belirlenmesi

In [ ]:
inertia = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(K_range, inertia, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Küme Sayısı (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Metodu - Optimum K Değeri')

axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Küme Sayısı (K)')
axes[1].set_ylabel('Silhouette Skoru')
axes[1].set_title('Silhouette Skoru - Optimum K Değeri')

plt.tight_layout()
plt.show()

In [ ]:
optimal_k = 2
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

print('K-Means Kümeleme Sonuçları:')
print(f'Küme sayısı: {optimal_k}')
print(f'Silhouette Skoru: {silhouette_score(X_scaled, kmeans_labels):.4f}')
print(f'Davies-Bouldin İndeksi: {davies_bouldin_score(X_scaled, kmeans_labels):.4f}')
print(f'Calinski-Harabasz Skoru: {calinski_harabasz_score(X_scaled, kmeans_labels):.4f}')

## Boyut İndirgeme ile Görselleştirme (PCA ve t-SNE)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='viridis', alpha=0.7, s=50, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Küme')
plt.title('K-Means Kümeleme Sonuçları (PCA ile 2 Boyutlu Gösterim)', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 (%{pca.explained_variance_ratio_[0] * 100:.1f} varyans)', fontsize=12)
plt.ylabel(f'PC2 (%{pca.explained_variance_ratio_[1] * 100:.1f} varyans)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=kmeans_labels, cmap='viridis', alpha=0.7, s=50, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Küme')
plt.title('K-Means Kümeleme Sonuçları (t-SNE ile 2 Boyutlu Gösterim)', fontsize=14, fontweight='bold')
plt.xlabel('t-SNE Boyut 1')
plt.ylabel('t-SNE Boyut 2')
plt.tight_layout()
plt.show()

## Silhouette Diyagramı

Her bir örneğin kendi kümesine ne kadar iyi uyduğunu gösteren detaylı silhouette diyagramı çizilmiştir.

In [ ]:
silhouette_ornek_degerleri = silhouette_samples(X_scaled, kmeans_labels)

plt.figure(figsize=(10, 7))
y_alt = 10

for i in range(optimal_k):
    kume_degerleri = silhouette_ornek_degerleri[kmeans_labels == i]
    kume_degerleri.sort()
    kume_boyutu = kume_degerleri.shape[0]
    y_ust = y_alt + kume_boyutu

    plt.fill_betweenx(np.arange(y_alt, y_ust), 0, kume_degerleri, alpha=0.7)
    plt.text(-0.05, y_alt + 0.5 * kume_boyutu, str(i))
    y_alt = y_ust + 10

plt.axvline(x=silhouette_score(X_scaled, kmeans_labels), color='red', linestyle='--', label='Ortalama Silhouette Skoru')
plt.xlabel('Silhouette Katsayısı')
plt.ylabel('Küme Etiketi')
plt.title('K-Means Silhouette Diyagramı')
plt.legend()
plt.tight_layout()
plt.show()

HİYERARŞİK KÜMELEME

In [ ]:
agg_clustering = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
agg_labels = agg_clustering.fit_predict(X_scaled)

print('Hiyerarşik Kümeleme (Agglomerative) Sonuçları:')
print(f'Küme sayısı: {optimal_k}')
print(f'Bağlantı yöntemi: ward')
print(f'Silhouette Skoru: {silhouette_score(X_scaled, agg_labels):.4f}')
print(f'Davies-Bouldin İndeksi: {davies_bouldin_score(X_scaled, agg_labels):.4f}')
print(f'Calinski-Harabasz Skoru: {calinski_harabasz_score(X_scaled, agg_labels):.4f}')

In [ ]:
plt.figure(figsize=(16, 8))
linked = linkage(X_scaled[:200], method='ward')
dendrogram(linked, orientation='top', distance_sort='descending', show_leaf_counts=False, truncate_mode='level', p=10)
plt.title('Hiyerarşik Kümeleme - Dendrogram (Ward Bağlantısı)', fontsize=14, fontweight='bold')
plt.xlabel('Veri Noktaları', fontsize=12)
plt.ylabel('Mesafe (Öklid)', fontsize=12)
plt.axhline(y=30, color='r', linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=agg_labels, cmap='plasma', alpha=0.7, s=50, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Küme')
plt.title('Hiyerarşik Kümeleme Sonuçları (PCA ile 2 Boyutlu Gösterim)', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 (%{pca.explained_variance_ratio_[0] * 100:.1f} varyans)', fontsize=12)
plt.ylabel(f'PC2 (%{pca.explained_variance_ratio_[1] * 100:.1f} varyans)', fontsize=12)
plt.tight_layout()
plt.show()

DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

In [ ]:
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)
distances = np.sort(distances[:, 4], axis=0)

plt.figure(figsize=(12, 5))
plt.plot(distances, 'b-', linewidth=2)
plt.title('k-Mesafe Grafiği (k=5) - eps Değeri Belirleme', fontsize=14, fontweight='bold')
plt.xlabel('Veri Noktaları (sıralanmış)', fontsize=12)
plt.ylabel('5. En Yakın Komşu Mesafesi', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
dbscan = DBSCAN(eps=2.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print('DBSCAN Kümeleme Sonuçları:')
print('eps: 2.5, min_samples: 5')
print(f'Oluşan küme sayısı: {n_clusters_db}')
print(f'Gürültü (aykırı değer) sayısı: {n_noise}')
print(f'Gürültü oranı: %{n_noise / len(dbscan_labels) * 100:.2f}')

if n_clusters_db > 1:
    print(f'Silhouette Skoru: {silhouette_score(X_scaled, dbscan_labels):.4f}')

In [ ]:
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=dbscan_labels, cmap='tab10', alpha=0.7, s=50, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Küme (-1 = Gürültü)')
plt.title('DBSCAN Kümeleme Sonuçları (PCA ile 2 Boyutlu Gösterim)', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 (%{pca.explained_variance_ratio_[0] * 100:.1f} varyans)', fontsize=12)
plt.ylabel(f'PC2 (%{pca.explained_variance_ratio_[1] * 100:.1f} varyans)', fontsize=12)
plt.tight_layout()
plt.show()

GAUSS KARIŞIM MODELİ (Gaussian Mixture Model)

In [ ]:
gmm = GaussianMixture(n_components=optimal_k, random_state=42)
gmm_labels = gmm.fit_predict(X_scaled)

print('Gauss Karışım Modeli (GMM) Sonuçları:')
print(f'Küme sayısı: {optimal_k}')
print(f'Silhouette Skoru: {silhouette_score(X_scaled, gmm_labels):.4f}')
print(f'Davies-Bouldin İndeksi: {davies_bouldin_score(X_scaled, gmm_labels):.4f}')
print(f'Calinski-Harabasz Skoru: {calinski_harabasz_score(X_scaled, gmm_labels):.4f}')

In [ ]:
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=gmm_labels, cmap='cool', alpha=0.7, s=50, edgecolors='k', linewidth=0.5)
plt.colorbar(scatter, label='Küme')
plt.title('Gauss Karışım Modeli Sonuçları (PCA ile 2 Boyutlu Gösterim)', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 (%{pca.explained_variance_ratio_[0] * 100:.1f} varyans)', fontsize=12)
plt.ylabel(f'PC2 (%{pca.explained_variance_ratio_[1] * 100:.1f} varyans)', fontsize=12)
plt.tight_layout()
plt.show()

## Yöntemlerin Karşılaştırılması

In [ ]:
karsilastirma = pd.DataFrame({
    'Yöntem': ['K-Means', 'Hiyerarşik (Agglomerative)', 'DBSCAN', 'Gauss Karışım Modeli'],
    'Silhouette Skoru': [
        silhouette_score(X_scaled, kmeans_labels),
        silhouette_score(X_scaled, agg_labels),
        silhouette_score(X_scaled, dbscan_labels) if n_clusters_db > 1 else np.nan,
        silhouette_score(X_scaled, gmm_labels)
    ],
    'Davies-Bouldin İndeksi': [
        davies_bouldin_score(X_scaled, kmeans_labels),
        davies_bouldin_score(X_scaled, agg_labels),
        davies_bouldin_score(X_scaled, dbscan_labels) if n_clusters_db > 1 else np.nan,
        davies_bouldin_score(X_scaled, gmm_labels)
    ]
})
karsilastirma.sort_values('Silhouette Skoru', ascending=False).reset_index(drop=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(karsilastirma['Yöntem'], karsilastirma['Silhouette Skoru'], color='#16a085')
axes[0].set_title('Silhouette Skoru Karşılaştırması')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(karsilastirma['Yöntem'], karsilastirma['Davies-Bouldin İndeksi'], color='#c0392b')
axes[1].set_title('Davies-Bouldin İndeksi Karşılaştırması')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## Küme Profillerinin Çıkarılması

En iyi performansı gösteren K-Means kümeleri, orijinal (ölçeklendirilmemiş) değişkenler üzerinden demografik ve harcama açısından profillenmiştir.

In [ ]:
profil_df = copy_df.copy()
profil_df['Kume'] = kmeans_labels

profil_ozet = profil_df.groupby('Kume')[['Age', 'Income', 'Total_Spending', 'Total_Children', 'Total_Purchases', 'Recency']].mean()
profil_ozet

In [ ]:
profil_ozet_normalize = (profil_ozet - profil_ozet.min()) / (profil_ozet.max() - profil_ozet.min())

plt.figure(figsize=(12, 6))
profil_ozet_normalize.T.plot(kind='bar', figsize=(12, 6), colormap='viridis')
plt.title('Kümelere Göre Normalize Edilmiş Ortalama Özellik Değerleri')
plt.ylabel('Normalize Değer')
plt.xticks(rotation=30)
plt.legend(title='Küme')
plt.tight_layout()
plt.show()

## Sonuç

Elbow ve silhouette analizleri iki kümenin veri yapısına en uygun ayrım olduğunu göstermektedir. K-Means ve Hiyerarşik Kümeleme benzer sonuçlar üretirken, DBSCAN yoğunluk temelli çalıştığı için bazı noktaları gürültü olarak işaretlemektedir. Küme profilleri incelendiğinde, kümelerden biri daha yüksek gelir ve harcama düzeyine sahip müşterileri, diğeri ise daha düşük gelir ve harcama düzeyine sahip müşterileri temsil etmektedir; bu bilgi pazarlama stratejilerinin segment bazlı özelleştirilmesinde kullanılabilir.